# Baby Food Science — Import Pipeline

**Steps:**
1. Fetch OpenAlex counts for all active topics (determines node sizes in the map)
2. Browse active topics and their claims
3. Run the import (fetches papers from OpenAlex into SQLite — one DB per topic)
4. Build the frontend JSON files (`universe.json` + per-topic `nodes/edges.json`)
5. Run the keyword pre-pass (fast scan linking papers to claims)
6. Inspect what was imported

**Architecture:** Each topic uses a single broad query to pull all relevant papers.
Claims (specific recommendations to evaluate) are stored separately and matched to
papers via keyword pre-pass, then optionally via local Ollama/Mistral AI.

AI enrichment (claim evaluation) is a separate optional step — skip it to get a working map immediately.

In [1]:
import sys, os, glob, sqlite3

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.abspath(os.path.join(BACKEND_DIR, "..", "data"))
OUT_DIR     = os.path.abspath(os.path.join(BACKEND_DIR, "..", "frontend", "public"))

sys.path.insert(0, BACKEND_DIR)
from import_openalex import TOPIC_QUERIES, CLAIMS, import_topic, fetch_all_topic_counts, keyword_prepass
from build_data import build_universe

# Backwards-compatible alias used in some cells below
PREDEFINED_TOPICS = TOPIC_QUERIES

print(f"Backend : {BACKEND_DIR}")
print(f"Data    : {DATA_DIR}")
print(f"Output  : {OUT_DIR}")
print(f"Active topics : {len(TOPIC_QUERIES)}")
print(f"Claims        : {len(CLAIMS)}")

Backend : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\backend
Data    : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data
Output  : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\frontend\public
Active topics : 4
Claims        : 12


## Step 1 — Fetch OpenAlex paper counts

Makes one lightweight API call per topic (no paper download) and saves counts to
`data/topic_counts.json`. These totals drive node size in the galaxy map.

Run this once upfront; re-run anytime to refresh the counts.

In [2]:
counts = fetch_all_topic_counts(out_path=os.path.join(DATA_DIR, "topic_counts.json"))
print(f"Fetched counts for {len(counts)} topics.")
top = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 by total OpenAlex papers:")
for k, v in top:
    print(f"  {k:45s}  {v:>8,d}")

  [  1/4] peanut_allergy                                   2,104
  [  2/4] vitamin_d                                       48,456
  [  3/4] water_infant                                   110,658
  [  4/4] cow_milk                                         8,861

Saved topic counts → C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\topic_counts.json
Fetched counts for 102 topics.
Top 5 by total OpenAlex papers:
  water_infant                                    110,658
  vitamin_d                                        48,456
  maternal_nutrition_pregnancy                     23,974
  breastfeeding                                    20,557
  childhood_obesity_diet                           10,944


## Step 2 — Browse active topics and claims

Shows all active topics, their broad fetch queries, and the specific claims
that will be evaluated against papers in each topic's database.

In [3]:
def db_paper_count(topic_key):
    path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(path):
        return None
    try:
        conn = sqlite3.connect(path)
        n = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

def db_claim_match_count(topic_key):
    path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(path):
        return {}
    try:
        conn = sqlite3.connect(path)
        rows = conn.execute(
            "SELECT claim_key, COUNT(*) FROM claim_evaluations WHERE keyword_match=1 GROUP BY claim_key"
        ).fetchall()
        conn.close()
        return dict(rows)
    except Exception:
        return {}

total_papers = 0
for topic_key, cfg in TOPIC_QUERIES.items():
    paper_count = db_paper_count(topic_key)
    status = f'{paper_count:,} papers' if paper_count is not None else '(not imported)'
    print(f'\n── {cfg["name"]} ({topic_key})')
    print(f'   Query  : "{cfg["query"]}"')
    print(f'   DB     : {status}')
    if paper_count:
        total_papers += paper_count

    match_counts = db_claim_match_count(topic_key)
    topic_claims = [(k, v) for k, v in CLAIMS.items() if v['topic'] == topic_key]
    for claim_key, claim_cfg in topic_claims:
        matched = match_counts.get(claim_key, 0)
        tag = f'[{matched} keyword matches]' if matched else '[pre-pass not run]'
        arrow = '→' if claim_cfg['direction'] == 'for' else '✗'
        print(f'   {arrow} [{claim_cfg["direction"]:7s}] {claim_cfg["claim"][:65]}  {tag}')

print(f'\n{len(TOPIC_QUERIES)} active topics, {total_papers:,} papers total across imported DBs')
print(f'{len(CLAIMS)} claims to evaluate')


── Peanut Allergy (peanut_allergy)
   Query  : "peanut allergy infant child"
   DB     : 0 papers
   → [for    ] Introducing peanuts at around 4 months of age reduces the risk of  [pre-pass not run]
   ✗ [against] Introducing peanuts at 4 months is not recommended or may increas  [pre-pass not run]
   → [for    ] Introducing peanuts at around 6 months of age reduces the risk of  [pre-pass not run]
   ✗ [against] Introducing peanuts at 6 months is not sufficient to prevent alle  [pre-pass not run]
   → [for    ] Delaying peanut introduction beyond 6 months is safe and acceptab  [pre-pass not run]
   ✗ [against] Delaying peanut introduction beyond 6 months increases the risk o  [pre-pass not run]

── Vitamin D (vitamin_d)
   Query  : "vitamin D infant child"
   DB     : 200 papers
   → [for    ] Vitamin D supplementation is recommended for breastfed infants to  [pre-pass not run]
   → [for    ] Adequate sun exposure can provide sufficient vitamin D for infant  [pre-pass not run]

── Wat

## Step 3 — Configure & import

Edit `TOPICS_TO_IMPORT` to select which topics to fetch.  
Use `list(TOPIC_QUERIES.keys())` to import all active topics.

`MAX_PAPERS` controls how many papers to fetch per topic from OpenAlex.  
`MIN_CITATIONS` filters out papers with fewer citations (0 = include everything).  
`RUN_PREPASS` automatically runs the keyword pre-pass after each import.

In [8]:
# ── configure here ──────────────────────────────────────────────────────────

TOPICS_TO_IMPORT = list(TOPIC_QUERIES.keys())   # all active topics

# Or pick specific ones:
# TOPICS_TO_IMPORT = ['peanut_allergy', 'vitamin_d']

MAX_PAPERS    = 200   # papers per topic from OpenAlex
MIN_CITATIONS = 0     # set higher (e.g. 5) to skip low-impact papers
SKIP_EXISTING = False  # skip topics whose DB already has papers
RUN_PREPASS   = True  # run keyword pre-pass immediately after each fetch

# ────────────────────────────────────────────────────────────────────────────

from import_openalex import BudgetExhaustedError

os.makedirs(DATA_DIR, exist_ok=True)

to_run = []
for key in TOPICS_TO_IMPORT:
    if key not in TOPIC_QUERIES:
        print(f'  [warn] unknown topic key: {key} — skipping')
        continue
    existing_count = db_paper_count(key)
    if SKIP_EXISTING and existing_count:
        print(f'  [skip] {key} — already has {existing_count} papers')
        continue
    to_run.append(key)

print(f'\nWill import {len(to_run)} topic(s): {", ".join(to_run) or "(none)"}')


Will import 4 topic(s): peanut_allergy, vitamin_d, water_infant, cow_milk


In [9]:
results = {}
for i, key in enumerate(to_run, 1):
    cfg = TOPIC_QUERIES[key]
    print(f'\n[{i}/{len(to_run)}] {cfg["name"]} ({key})')
    print(f'  Query: "{cfg["query"]}"')
    try:
        db_path = import_topic(
            key,
            max_results=MAX_PAPERS,
            min_citations=MIN_CITATIONS,
            run_prepass=RUN_PREPASS,
        )
        count = db_paper_count(key) or 0
        results[key] = count
        print(f'  → {count} papers in database')
    except BudgetExhaustedError as e:
        print(f'\n[BUDGET EXHAUSTED] {e}')
        print(f'Re-run this cell tomorrow to continue from: {key}')
        break
    except Exception as e:
        print(f'  [error] {e}')
        results[key] = 0

print(f'\nDone. {sum(results.values()):,} papers imported across {len(results)} topics.')


[1/4] Peanut Allergy (peanut_allergy)
  Query: "peanut allergy infant child"
[import] Topic: peanut_allergy
[import] Query: peanut allergy infant child
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\papers_peanut_allergy.db
[import] Existing papers: 200
[import] Total in OpenAlex: 2,104
  Fetched 200 papers total.   
[import] Done. Inserted: 0, Skipped: 200, Citations: 0
[import] Running keyword pre-pass...
  [prepass] 0 (paper, claim) keyword matches across 200 papers
  → 200 papers in database

[2/4] Vitamin D (vitamin_d)
  Query: "vitamin D infant child"
[import] Topic: vitamin_d
[import] Query: vitamin D infant child
[import] DB:    C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data\papers_vitamin_d.db
[import] Existing papers: 200
[import] Total in OpenAlex: 48,456
  Fetched 200 papers total.   
[import] Done. Inserted: 190, Skipped: 10, Citations: 113
[import] Running keyword pre-pass...
  [prepass] 491 (paper, claim) keyword matches across 390 papers
  → 3

## Step 4 — Build frontend JSON

Reads all databases in `data/` and writes:
- `frontend/public/universe.json` — the top-level galaxy map
- `frontend/public/data/<topic>/nodes.json`
- `frontend/public/data/<topic>/edges.json`

Refresh the browser after this runs.

In [13]:
os.makedirs(OUT_DIR, exist_ok=True)
build_universe(DATA_DIR, OUT_DIR)
print('\nFrontend JSON built. Refresh http://localhost:3000 to see the updated map.')

[build] Found 102 databases
[build] Processing allergy_prevention_diet...
  [ok] allergy_prevention_diet: 200 papers, 103 citations
[build] Processing appetite_regulation...
  [ok] appetite_regulation: 30 papers, 10 citations
[build] Processing baby_food_labelling...
  [ok] baby_food_labelling: 200 papers, 314 citations
[build] Processing baby_food_marketing...
  [ok] baby_food_marketing: 200 papers, 351 citations
[build] Processing baby_led_weaning...
  [ok] baby_led_weaning: 200 papers, 386 citations
[build] Processing bpa_packaging_infant...
  [ok] bpa_packaging_infant: 162 papers, 14 citations
[build] Processing brain_development_nutrition...
  [ok] brain_development_nutrition: 78 papers, 16 citations
[build] Processing breastfeeding...
  [ok] breastfeeding: 219 papers, 37 citations
[build] Processing breastfeeding_difficulties...
  [ok] breastfeeding_difficulties: 159 papers, 42 citations
[build] Processing calcium_bone_infant...
  [ok] calcium_bone_infant: 200 papers, 66 citation

## Step 5 — Keyword pre-pass

Fast scan of all papers against each claim's keyword hints. Populates the
`claim_evaluations` table in each topic DB. Run this after importing to see which
papers are candidates for each claim before running the full AI evaluation.

Skip this if you already set `RUN_PREPASS = True` in Step 3 — it was run automatically.

In [11]:
for topic_key in TOPIC_QUERIES:
    db_path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(db_path):
        print(f'  [skip] {topic_key} — not imported yet')
        continue
    print(f'\n── {TOPIC_QUERIES[topic_key]["name"]}')
    keyword_prepass(db_path, topic_key=topic_key)

# Show summary of matches per claim
print('\n── Match counts per claim:')
for topic_key in TOPIC_QUERIES:
    db_path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(db_path):
        continue
    conn = sqlite3.connect(db_path)
    topic_claims = [k for k, v in CLAIMS.items() if v['topic'] == topic_key]
    for claim_key in topic_claims:
        n = conn.execute(
            "SELECT COUNT(*) FROM claim_evaluations WHERE claim_key=? AND keyword_match=1",
            (claim_key,)
        ).fetchone()[0]
        print(f'  {claim_key:35s}  {n:>4} papers matched')
    conn.close()


── Peanut Allergy
  [prepass] 0 (paper, claim) keyword matches across 200 papers

── Vitamin D
  [prepass] 0 (paper, claim) keyword matches across 390 papers

── Water for Infants
  [prepass] 0 (paper, claim) keyword matches across 200 papers

── Cow's Milk Introduction
  [prepass] 0 (paper, claim) keyword matches across 200 papers

── Match counts per claim:
  peanut_intro_4m_for                    63 papers matched
  peanut_intro_4m_against               127 papers matched
  peanut_intro_6m_for                    89 papers matched
  peanut_intro_6m_against                83 papers matched
  peanut_intro_later_for                 33 papers matched
  peanut_intro_later_against             44 papers matched
  vitamin_d_supplement_for              314 papers matched
  vitamin_d_sun_sufficient              177 papers matched
  water_delay_for                        57 papers matched
  water_after_6m_ok                      78 papers matched
  cow_milk_12m_for                       41 pap

## Step 6 — Inspect imported data

In [12]:
dbs = sorted(glob.glob(os.path.join(DATA_DIR, 'papers_*.db')))
if not dbs:
    print('No databases found. Run Step 3 first.')
else:
    total = 0
    print(f'{"Topic":45s} {"Papers":>8s} {"Claim matches":>14s} {"Year range":>12s}')
    print('-' * 85)
    for db_path in dbs:
        key = os.path.basename(db_path).replace('papers_', '').replace('.db', '')
        conn = sqlite3.connect(db_path)
        try:
            rows = conn.execute(
                'SELECT COUNT(*) as n, MIN(year) as yr_min, MAX(year) as yr_max FROM papers'
            ).fetchone()
            n, yr0, yr1 = rows
            yr_range = f'{yr0}–{yr1}' if yr0 and yr1 else 'unknown'

            try:
                claim_matches = conn.execute(
                    "SELECT COUNT(*) FROM claim_evaluations WHERE keyword_match=1"
                ).fetchone()[0]
                match_str = f'{claim_matches:,}'
            except Exception:
                match_str = '—'

            name = TOPIC_QUERIES.get(key, {}).get('name', key)
            print(f'{name:45s} {n:>8,d} {match_str:>14s} {yr_range:>12s}')
            total += n
        except Exception as e:
            print(f'{key}: error — {e}')
        finally:
            conn.close()
    print('-' * 85)
    print(f'{"TOTAL":45s} {total:>8,d}')

Topic                                           Papers  Claim matches   Year range
-------------------------------------------------------------------------------------
allergy_prevention_diet                            200              —    1999–2025
appetite_regulation                                 30              —    2002–2026
baby_food_labelling                                200              —    1994–2026
baby_food_marketing                                200              —    2001–2025
baby_led_weaning                                   200              —    1930–2024
bpa_packaging_infant                               162              —    2005–2026
brain_development_nutrition                         78              —    2009–2026
breastfeeding                                      219              —    1985–2025
breastfeeding_difficulties                         159              —    1999–2026
calcium_bone_infant                                200              —    1976–2025
c

## Optional — AI claim evaluation

Run this to evaluate each keyword-matched paper against its claims using a local
Ollama/Mistral LLM. Results are stored in the `claim_evaluations` table.

Requires Ollama running locally: `ollama serve && ollama pull mistral`

Run the keyword pre-pass (Step 5) first to reduce the number of LLM calls.

In [ ]:
# ── configure ────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = 'mistral'
ONLY_KEYWORD_MATCHED = True   # True = only evaluate papers that passed keyword pre-pass
# ─────────────────────────────────────────────────────────────────────────────

from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

EVAL_SYSTEM = (
    "You are a pediatric nutrition research assistant. "
    "Respond ONLY with valid JSON. Be concise."
)

EVAL_PROMPT = """\
Does this paper support, contradict, or is it irrelevant to the following claim?

Claim: {claim}

Paper title: {title}
Abstract: {abstract}

Respond with ONLY this JSON (no markdown):
{{
  "stance": "<supports | contradicts | neutral | irrelevant>",
  "confidence": <0-100 integer>,
  "summary": "<one sentence: what the paper says about this claim>"
}}"""

import json, re, time, sqlite3

def eval_paper_claim(client, model, paper, claim_cfg):
    prompt = EVAL_PROMPT.format(
        claim=claim_cfg['claim'],
        title=paper['title'] or '(no title)',
        abstract=(paper['abstract'] or '')[:1500],
    )
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": EVAL_SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.1, max_tokens=200,
            )
            text = resp.choices[0].message.content.strip()
            text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.MULTILINE)
            text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE)
            data = json.loads(text)
            stance = data.get('stance', 'irrelevant')
            if stance not in ('supports', 'contradicts', 'neutral', 'irrelevant'):
                stance = 'irrelevant'
            return {
                'stance': stance,
                'confidence': max(0, min(100, int(data.get('confidence', 50)))),
                'summary': str(data.get('summary', ''))[:400],
            }
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
    return None

for topic_key in TOPIC_QUERIES:
    db_path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(db_path):
        continue
    topic_claims = {k: v for k, v in CLAIMS.items() if v['topic'] == topic_key}
    if not topic_claims:
        continue

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    for claim_key, claim_cfg in topic_claims.items():
        if ONLY_KEYWORD_MATCHED:
            papers = conn.execute(
                """SELECT p.paperId, p.title, p.abstract
                   FROM papers p
                   JOIN claim_evaluations ce ON ce.paperId = p.paperId
                   WHERE ce.claim_key=? AND ce.keyword_match=1 AND ce.llm_stance IS NULL""",
                (claim_key,)
            ).fetchall()
        else:
            papers = conn.execute(
                """SELECT p.paperId, p.title, p.abstract
                   FROM papers p
                   LEFT JOIN claim_evaluations ce ON ce.paperId=p.paperId AND ce.claim_key=?
                   WHERE ce.llm_stance IS NULL OR ce.paperId IS NULL""",
                (claim_key,)
            ).fetchall()

        if not papers:
            print(f'  [skip] {claim_key} — no pending papers')
            continue

        print(f'\n── {claim_key} ({len(papers)} papers to evaluate)')
        done = 0
        for paper in papers:
            result = eval_paper_claim(client, OLLAMA_MODEL, dict(paper), claim_cfg)
            if result:
                conn.execute(
                    """INSERT OR REPLACE INTO claim_evaluations
                       (paperId, claim_key, keyword_match, llm_stance, llm_confidence, llm_summary)
                       VALUES (?, ?, 1, ?, ?, ?)""",
                    (paper['paperId'], claim_key, result['stance'],
                     result['confidence'], result['summary'])
                )
                done += 1
        conn.commit()
        print(f'  {done}/{len(papers)} papers evaluated for {claim_key}')

    conn.close()

print('\nDone. Re-run Step 4 to rebuild frontend JSON with claim data.')